In [6]:
import os
import re
import glob
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from stop_words_custom import stop_word_list
from unpack_documents import load_documents, preprocess

# Directory containing .txt files
TEXT_DIR = "text" 

# Directory containing other metadata
DATA_DIR = "data"

In [1]:
male_pronouns = ["he", "hed", "he'd", "he'll", "hes", "he's", "him", "himself", "his"]
female_pronouns = ["she", "shed", "she'd", "she'll", "shes", "she's", "her", "herself", "hers"]

## Load documents

In [7]:
filepaths = glob.glob(os.path.join(TEXT_DIR, "*.txt"))

corpus, filenames = load_documents(filepaths)

Loaded 762 documents


## Preprocessing documents (stemming)

In [8]:
stemmer = PorterStemmer()

corpus_stemmed = [preprocess(doc, stemmer) for doc in corpus]

## Bag of words

In [9]:
vectorizer = CountVectorizer(
    strip_accents='unicode',
    stop_words=stop_word_list,
    max_df=1.0,
    min_df=5
)

X = vectorizer.fit_transform(corpus_stemmed)

print("Matrix shape:", X.shape)

/Users/mjl09005/Library/CloudStorage/OneDrive-UniversityofConnecticut/gvp/docx_env/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['articl', 'mon'] not in stop_words.
  warnings.warn(


Matrix shape: (762, 14358)


In [10]:
feature_names = vectorizer.get_feature_names_out()

bow_df = pd.DataFrame(X.toarray(), columns=feature_names, index=filenames)

bow_df.head()

,aa,aaron,aback,abalon,abandon,abash,abat,abattoir,abbey,abbi,...,zigzag,zillion,zinc,zip,zipper,zombi,zone,zoo,zoom,zu
412 Brisk Money.txt,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
164 Specimen 313.txt,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
270 When We Were Heroes.txt,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
720 Tie A Yellow Ribbon.txt,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
176 Apologue.txt,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [12]:
word_counts = bow_df.sum(axis=0).sort_values(ascending=False)

word_counts.head()

he      70200
she     59722
her     55972
they    29427
him     20985
dtype: int64

In [15]:
sum_male = 0
sum_female = 0

for pronoun in male_pronouns:
    sum_male += word_counts.get(pronoun, 0)

for pronoun in female_pronouns:
    sum_female += word_counts.get(pronoun, 0)

print(f"Total male pronouns: {sum_male}")
print(f"Total female pronouns: {sum_female}")

Total male pronouns: 94050
Total female pronouns: 118077


In [18]:
bow_df["male_total"] = sum(bow_df[male_pronouns])

KeyError: '[\'hed\', "he\'d", "he\'ll", \'hes\', "he\'s", \'his\'] not in index'